# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring a dataset using the `mlcroissant` library. We demonstrate best practices for programmatically exploring and processing a Croissant-packaged, FAIR dataset.

### Dataset Source
The dataset is described by a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure mlcroissant is installed (run if not already installed; may require a restart)
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# URL for the Croissant schema
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's get an overview of the record sets (`@id`s), fields, and columns present in the dataset.

We will enumerate all available record sets, along with their field and column `@id`s, for later reference.

In [ ]:
# List record sets and their fields by @id
record_set_entities = dataset.metadata.record_sets
if not record_set_entities:
    print("No record sets specified in metadata.")
else:
    for rs in record_set_entities:
        print(f"Record set: {rs.name} (@id={rs.id})")
        print("  Fields:")
        for f in rs.fields:
            print(f"    - {f.name} (@id={f.id}, dataType={getattr(f, 'data_type', None)})")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Replace the below values with the actual `@id` from the output above. For this dataset, let's proceed with the main survey record set, which is commonly structured as the main table.

**Note:** In Croissant datasets following the MLCommons spec, a main record set often uses the name 'Main' or similar. Adjust as needed for your dataset.

In [ ]:
# Find the main record set, fallback to the first if not present
record_sets = dataset.metadata.record_sets
if record_sets:
    # Identify likely main tabular set (common naming: 'Survey', 'Main', etc.)
    main_rs = None
    for rs in record_sets:
        if 'main' in rs.name.lower() or 'survey' in rs.name.lower():
            main_rs = rs
            break
    if main_rs is None:
        main_rs = record_sets[0]
        print(f"Chose record set: {main_rs.name} (@id={main_rs.id}) as main table.")
    record_set_ids = [rs.id for rs in record_sets]
else:
    raise ValueError('No record sets found in metadata.')

In [ ]:
# Load all record sets into DataFrames by @id
dataframes = {}
for rs in record_sets:
    # Each records() generator yields dicts using field @id as keys
    try:
        records = list(dataset.records(record_set=rs.id))
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"Loaded DataFrame for {rs.name} (@id={rs.id}), shape={df.shape}")
    except Exception as e:
        print(f"Could not load {rs.name} (@id={rs.id}): {e}")

# Example: Preview columns for the main table
main_df = dataframes[main_rs.id]
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Let's apply standard EDA steps: filtering, normalization, and grouping.

We'll select a numeric field for demonstration (for instance, the GAD-7 sum score, often named `gad_7_score` or similar in mental health surveys). Adjust field `@id` as per the actual column names listed above.

In [ ]:
# Pick a numeric field (adjust according to actual field @id in the DataFrame)
# Usually named e.g. gad_7_score or similar; you can print main_df.columns to inspect
possible_numeric_ids = [col for col in main_df.columns if any(key in col.lower() for key in ['score', 'age', 'phq', 'gad', 'psq', 'sum'])]
if possible_numeric_ids:
    numeric_field = possible_numeric_ids[0]
    print(f"Using numeric field: {numeric_field}")
else:
    print("No likely numeric score columns found. Please assign numeric_field manually.")
    numeric_field = None

if numeric_field is not None:
    threshold = main_df[numeric_field].dropna().mean()
    filtered_df = main_df[main_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}, count: {len(filtered_df)}")

    normalized_col = f"{numeric_field}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, normalized_col]].head())

    # Try grouping by a demographic, e.g., 'gender' or 'village' (adjust as per real columns)
    possible_group_fields = [col for col in main_df.columns if any(grp in col.lower() for grp in ["gender", "village", "age", "education"])]
    if possible_group_fields:
        group_field = possible_group_fields[0]
        print(f"Grouping by: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print("Grouped means:")
        print(grouped_df.head())
    else:
        print("No obvious group field available.")

## 5. Visualization
Let's plot the normalized numeric field's distribution and the group means visualized by the grouping demographic (if present).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[normalized_col].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of normalized {numeric_field} (filtered)")
    plt.xlabel(f"{numeric_field} (normalized)")
    plt.ylabel("Count")
    plt.show()

    # Grouped bar chart if group_field available
    if 'grouped_df' in locals() and not grouped_df.empty:
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore a FAIR dataset described with a Croissant schema using the `mlcroissant` library. We:

- Loaded dataset metadata and inspected record set/field structure using `@id` references.
- Loaded all data records using the `records(record_set=...)` method.
- Filtered, normalized, and grouped numeric variables for EDA.
- Visualized field distributions and group-level statistics.

This approach ensures reproducibility, clear schema-driven references to data fields, and efficient exploratory analysis in compliance with FAIR principles.

For further analysis, see the provided field documentation and leverage additional columns as needed.